# UN

In [1]:
import numpy as np
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from scipy.io import loadmat
import pickle
import pandas as pd

ckpt = 'datasets/un_DETM/min_df_100/detm_undebates_K_50_Htheta_800_Optim_adam_Clip_0.0_ThetaAct_relu_Lr_0.005_Bsz_1000_RhoSize_300_L_3_minDF_100_trainEmbeddings_1'
beta = loadmat(ckpt + '_beta.mat')['values']  # K x T x V

with open('datasets/un_DETM/min_df_100/vocab.pkl', 'rb') as f:
    vocab = pickle.load(f)

K, T, V = beta.shape

top_words_per_time = []

for t in range(T):  # anos
    topics_t = []
    for k in range(K):  # tópicos
        top_indices = beta[k, t, :].argsort()[::-1][:10]
        topics_t.append([vocab[i] for i in top_indices])
    top_words_per_time.append(topics_t)

tokens_raw = loadmat(
    'datasets/un_DETM/min_df_100/bow_tr_tokens.mat'
)['tokens']

tokens_raw = tokens_raw.squeeze() 

tokens = []

for doc in tokens_raw:
    d = doc.squeeze()

    if isinstance(d, (int, np.integer)):
        tokens.append([int(d)])
    else:
        tokens.append(d.astype(int).tolist())

tokens_fixed = []

for doc in tokens:
    if isinstance(doc, (int, np.integer)):
        tokens_fixed.append([int(doc)])
    else:
        tokens_fixed.append(list(map(int, doc)))

tokens = tokens_fixed

texts = [[vocab[w] for w in doc] for doc in tokens]

mat = loadmat('datasets/un_DETM/min_df_100/bow_tr_timestamps.mat')
timestamps_tr = mat['timestamps'].squeeze().tolist()

from collections import defaultdict

texts_by_time = defaultdict(list)
for doc, t in zip(texts, timestamps_tr):
    texts_by_time[t].append(doc)

In [ ]:
from functions import *
import pandas as pd
from gensim import corpora
from gensim.models import CoherenceModel
import numpy as np

results = []

for t in sorted(texts_by_time.keys()):
    texts_t = texts_by_time[t]
    texts_t_clean = [[str(w) for w in doc] for doc in texts_t]

    topics_t = top_words_per_time[t]
    topics_t_clean = [[str(w) for w in topic] for topic in topics_t]

    dictionary_t = corpora.Dictionary(texts_t_clean)

    topics_t_filtered = [
        [w for w in topic if w in dictionary_t.token2id]
        for topic in topics_t_clean
    ]
    topics_t_filtered = [topic for topic in topics_t_filtered if len(topic) > 0]

    if len(topics_t_filtered) == 0:
        results.append({
            'time_id': t,
            'cv': np.nan,
            'diversity': np.nan,
            'topic_quality': np.nan
        })
        continue

    cv_model = CoherenceModel(topics=topics_t_filtered, texts=texts_t_clean, dictionary=dictionary_t, coherence='c_v')
    cv = float(cv_model.get_coherence())

    diversity = topic_diversity(topics_t_filtered)
    topic_quality = cv * diversity if diversity is not np.nan else np.nan

    results.append({
        'time_id': t,
        'cv': cv,
        'diversity': diversity,
        'topic_quality': topic_quality
    })

df_metrics = pd.DataFrame(results)

try:
    import pickle
    time_list = pickle.load(open('datasets/un_DETM/min_df_100/timestamps.pkl', 'rb'))
    df_metrics['year'] = df_metrics['time_id'].apply(lambda t: time_list[t])
except:
    df_metrics['year'] = df_metrics['time_id']

df_metrics = df_metrics.sort_values('year').reset_index(drop=True)

df_metrics.head()

,time_id,cv,diversity,topic_quality,year
0,0,0.521345,0.998066,0.520337,1970
1,1,0.521139,0.998779,0.520502,1971
2,2,0.526703,0.998514,0.525920,1972
3,3,0.547183,0.998551,0.546390,1973
4,4,0.534033,0.998168,0.533055,1974


In [53]:
df_metrics_avg = df_metrics.groupby('year').agg({
    'cv': 'mean',
    'diversity': 'mean',
    'topic_quality': 'mean'
}).reset_index()

df_metrics_avg.head()

,year,cv,diversity,topic_quality
0,1970,0.521345,0.998066,0.520337
1,1971,0.521139,0.998779,0.520502
2,1972,0.526703,0.998514,0.525920
3,1973,0.547183,0.998551,0.546390
4,1974,0.534033,0.998168,0.533055


In [54]:
df_metrics_avg.to_csv('datasets/un_detm_qualityscore.csv', index=False)

# State of Union

In [ ]:
import numpy as np
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from scipy.io import loadmat
import pickle
import pandas as pd

ckpt = 'datasets/stateofunion_DETM/min_df_10/detm_stateofunion_K_50_Htheta_800_Optim_adam_Clip_0.0_ThetaAct_relu_Lr_0.005_Bsz_1000_RhoSize_300_L_3_minDF_10_trainEmbeddings_1'
beta = loadmat(ckpt + '_beta.mat')['values']  # K x T x V

with open('datasets/stateofunion_DETM/min_df_10/vocab.pkl', 'rb') as f:
    vocab = pickle.load(f)

K, T, V = beta.shape

top_words_per_time = []

for t in range(T):  # anos
    topics_t = []
    for k in range(K):  # tópicos
        top_indices = beta[k, t, :].argsort()[::-1][:10]
        topics_t.append([vocab[i] for i in top_indices])
    top_words_per_time.append(topics_t)

tokens_raw = loadmat(
    'datasets/stateofunion_DETM/min_df_10/bow_tr_tokens.mat'
)['tokens']

tokens_raw = tokens_raw.squeeze() 

tokens = []

for doc in tokens_raw:
    d = doc.squeeze()

    if isinstance(d, (int, np.integer)):
        tokens.append([int(d)])
    else:
        tokens.append(d.astype(int).tolist())

tokens_fixed = []

for doc in tokens:
    if isinstance(doc, (int, np.integer)):
        tokens_fixed.append([int(doc)])
    else:
        tokens_fixed.append(list(map(int, doc)))

tokens = tokens_fixed

texts = [[vocab[w] for w in doc] for doc in tokens]

mat = loadmat('datasets/stateofunion_DETM/min_df_10/bow_tr_timestamps.mat')
timestamps_tr = mat['timestamps'].squeeze().tolist()

from collections import defaultdict

texts_by_time = defaultdict(list)
for doc, t in zip(texts, timestamps_tr):
    texts_by_time[t].append(doc)

In [ ]:
from functions import *
import pandas as pd
from gensim import corpora
from gensim.models import CoherenceModel
import numpy as np

results = []

for t in sorted(texts_by_time.keys()):
    texts_t = texts_by_time[t]
    texts_t_clean = [[str(w) for w in doc] for doc in texts_t]

    topics_t = top_words_per_time[t]
    topics_t_clean = [[str(w) for w in topic] for topic in topics_t]

    dictionary_t = corpora.Dictionary(texts_t_clean)

    topics_t_filtered = [
        [w for w in topic if w in dictionary_t.token2id]
        for topic in topics_t_clean
    ]
    topics_t_filtered = [topic for topic in topics_t_filtered if len(topic) > 0]

    if len(topics_t_filtered) == 0:
        results.append({
            'time_id': t,
            'cv': np.nan,
            'diversity': np.nan,
            'topic_quality': np.nan
        })
        continue

    cv_model = CoherenceModel(topics=topics_t_filtered, texts=texts_t_clean, dictionary=dictionary_t, coherence='c_v')
    cv = float(cv_model.get_coherence())

    diversity = topic_diversity(topics_t_filtered)
    topic_quality = cv * diversity if diversity is not np.nan else np.nan

    results.append({
        'time_id': t,
        'cv': cv,
        'diversity': diversity,
        'topic_quality': topic_quality
    })

df_metrics = pd.DataFrame(results)

try:
    import pickle
    time_list = pickle.load(open('datasets/stateofunion_DETM/min_df_10/timestamps.pkl', 'rb'))
    df_metrics['year'] = df_metrics['time_id'].apply(lambda t: time_list[t])
except:
    df_metrics['year'] = df_metrics['time_id']

df_metrics = df_metrics.sort_values('year').reset_index(drop=True)

df_metrics.head()

,time_id,cv,npmi,umass,diversity,topic_quality,year
0,0,0.600951,NaN,NaN,0.996914,0.599096,1
1,1,0.680577,NaN,NaN,0.996881,0.678454,2
2,2,0.623814,NaN,NaN,0.996976,0.621927,3
3,3,0.550207,NaN,NaN,1.000000,0.550207,4
4,4,0.785614,NaN,NaN,1.000000,0.785614,5


In [49]:
df_metrics_avg = df_metrics.groupby('year').agg({
    'cv': 'mean',
    'diversity': 'mean',
    'topic_quality': 'mean'
}).reset_index()

df_metrics_avg.head()

,year,cv,diversity,topic_quality
0,1,0.600951,0.996914,0.599096
1,2,0.680577,0.996881,0.678454
2,3,0.623814,0.996976,0.621927
3,4,0.550207,1.000000,0.550207
4,5,0.785614,1.000000,0.785614


In [ ]:
df_metrics_avg.to_csv('datasets/stateofunion_detm_qualityscore.csv', index=False)